In [25]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix, classification_report


# Load data

In [26]:
df = pd.read_csv('labeled_motion_features.csv')

# Predict future alert (break leakage)

In [27]:
df["future_high_alert"] = df["high_alert"].shift(-1)
df = df.dropna()

# Time-aware train-test split


In [28]:
split_idx = int(0.7 * len(df))

train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

X_train = train_df[["mean_motion", "variance_motion"]]
y_train = train_df["future_high_alert"]

X_test = test_df[["mean_motion", "variance_motion"]]
y_test = test_df["future_high_alert"]

# Train model

In [29]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(max_depth=5, random_state=42)

# PREDICTION WITH LOWER THRESHOLD

In [31]:
y_prob = model.predict_proba(X_test)[:, 1]
y_prob

array([0.12787513, 0.15187513, 0.01297835, ..., 0.00095668, 0.0010142 ,
       0.0010142 ], shape=(1161,))

# Lower threshold to improve recall

In [32]:
THRESHOLD = 0.4
y_pred_raw = (y_prob >= THRESHOLD).astype(int)

# TEMPORAL OR SMOOTHING

In [34]:
WINDOW = 3
y_pred = []

for i in range(len(y_pred_raw)):
    if i < WINDOW:
        y_pred.append(y_pred_raw[i])
    else:
        y_pred.append(int(any(y_pred_raw[i-WINDOW:i+1])))

y_pred = np.array(y_pred)

# Evaluation

In [38]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:\n", cm)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Confusion Matrix:
 [[593 176]
 [ 19 373]]

Classification Report:

              precision    recall  f1-score   support

         0.0       0.97      0.77      0.86       769
         1.0       0.68      0.95      0.79       392

    accuracy                           0.83      1161
   macro avg       0.82      0.86      0.83      1161
weighted avg       0.87      0.83      0.84      1161

